# NumPy

NumPy provides the `ndarray`: a multi-dimensional array stored in contiguous memory and operated on by C and Fortran routines. It is the numerical foundation of scientific Python and of every quantum simulation library.

This notebook accompanies **Lesson 7** of *Python Programming for Quantum Technology I*. Topics covered:
- Arrays vs Python lists
- Representing qubit states as complex arrays
- Array operations and broadcasting
- Matrices and matrix multiplication
- Reshaping, slicing, and the Kronecker product

In [ ]:
import numpy as np

---
## 1. Arrays vs Python Lists

In [ ]:
# Python list: + means concatenation, * means repetition
a_list = [1, 2, 3]
b_list = [4, 5, 6]
print("List + :", a_list + b_list)    # [1, 2, 3, 4, 5, 6]
print("List * 2:", a_list * 2)        # [1, 2, 3, 1, 2, 3]

# NumPy array: + is element-wise addition, * is element-wise multiplication
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print("Array +:", a + b)              # [5 7 9]
print("Array *:", a * b)              # [ 4 10 18]

In [ ]:
print(np.zeros(4))                  # [0. 0. 0. 0.]
print(np.ones(3))                   # [1. 1. 1.]
print(np.eye(3))                    # 3x3 identity
print(np.arange(0, 8))             # [0 1 2 3 4 5 6 7]
print(np.linspace(0, 1, 5))        # [0.   0.25 0.5  0.75 1.  ]

In [ ]:
# dtype determines the element type
real_arr    = np.array([1.0, 2.0, 3.0])
complex_arr = np.array([1+0j, 0+1j, -1+0j], dtype=complex)

print(real_arr.dtype)      # float64
print(complex_arr.dtype)   # complex128

# shape, ndim, size
v = np.array([1, 0, 0, 0])
M = np.array([[1, 0], [0, 1]])

print(f"v: shape={v.shape}, ndim={v.ndim}, size={v.size}")
print(f"M: shape={M.shape}, ndim={M.ndim}, size={M.size}")

### Exercise 1.1

1. Create a 1D array of 8 evenly spaced angles from $0$ to $2\pi$ (inclusive).
2. Create a complex array representing the 8th roots of unity: $e^{2\pi i k/8}$ for $k = 0, 1, \ldots, 7$.
3. Print the real parts, imaginary parts, and moduli. Verify that all moduli equal 1.
4. Print the sum of all 8 roots. It should be (approximately) zero.

In [ ]:
import numpy as np

# Your code here


---
## 2. Representing a Qubit State

A qubit state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ is a 1D complex array of length 2.

In [ ]:
# Standard basis states
ket_0    = np.array([1, 0], dtype=complex)
ket_1    = np.array([0, 1], dtype=complex)

# Equal superposition states
ket_plus  = np.array([1,  1], dtype=complex) / np.sqrt(2)
ket_minus = np.array([1, -1], dtype=complex) / np.sqrt(2)

print("|0⟩:", ket_0)
print("|+⟩:", ket_plus.round(4))

In [ ]:
def normalize(psi):
    """Return a normalized copy of a complex state vector."""
    norm = np.linalg.norm(psi)
    return psi / norm

raw = np.array([3, 4], dtype=complex)
psi = normalize(raw)

print("Normalized:", psi)
print("Norm:", np.linalg.norm(psi).round(10))   # 1.0

In [ ]:
# Probabilities: |alpha_k|^2 for each component
psi = np.array([0.6, 0.8j])
probs = np.abs(psi)**2

print("Probabilities:", probs)     # [0.36 0.64]
print("Sum:", probs.sum())         # 1.0

In [ ]:
# Inner product <phi|psi> = phi.conj() @ psi
phi = np.array([1, 0], dtype=complex)   # |0⟩
psi = np.array([0.6, 0.8], dtype=complex)

print("<0|psi>:", phi.conj() @ psi)     # (0.6+0j)
print("<psi|psi>:", psi.conj() @ psi)   # (1+0j)  always 1 if normalized

# Orthogonality: <+|-> = 0
print("<+|->:", ket_plus.conj() @ ket_minus)   # 0

In [ ]:
# Multi-qubit state: equal superposition over all 3-qubit basis states
n = 3
psi = np.ones(2**n, dtype=complex) / np.sqrt(2**n)

probs = np.abs(psi)**2
labels = [f"|{format(k, f'0{n}b')}⟩" for k in range(2**n)]

print(f"{'State':<8} {'Amplitude':>12} {'Probability':>13}")
print("-" * 35)
for label, amp, prob in zip(labels, psi, probs):
    print(f"{label:<8} {amp.real:>12.4f} {prob:>13.4f}")
print(f"{'Total':>21} {probs.sum():>13.4f}")

### Exercise 2.1

Consider the two-qubit state:

$$|\psi\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle) \quad \text{(Bell state } |\Phi^+\rangle\text{)}$$

1. Represent this as a NumPy array over the 2-qubit basis $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$.
2. Compute and print the probability of each basis state.
3. Verify normalization.
4. Compute the inner product $\langle\Phi^+|\Phi^+\rangle$. It should equal 1.
5. Construct the $|\Phi^-\rangle = \frac{1}{\sqrt{2}}(|00\rangle - |11\rangle)$ state and compute $\langle\Phi^+|\Phi^-\rangle$. It should equal 0.

In [ ]:
import numpy as np

# Your code here


---
## 3. Array Operations and Broadcasting

In [ ]:
# Element-wise arithmetic
a = np.array([1, 2, 3, 4])
b = np.array([5, 6, 7, 8])

print(a + b)    # [ 6  8 10 12]
print(a * b)    # [ 5 12 21 32]
print(a ** 2)   # [ 1  4  9 16]

In [ ]:
# Universal functions: applied to every element, no Python loop
theta = np.linspace(0, 2*np.pi, 9)

print("Angles (deg):", np.degrees(theta).round(1))
print("sin:", np.sin(theta).round(4))
print("cos:", np.cos(theta).round(4))

# Complex exponentials: the building block of the quantum Fourier transform
roots = np.exp(1j * theta)
print("\ne^(i*theta):")
for k, z in enumerate(roots):
    print(f"  k={k}: {z.real:+.4f} {z.imag:+.4f}j  |z|={abs(z):.4f}")

In [ ]:
# Broadcasting: a global phase applied to a state vector
psi = np.array([0.6, 0.8], dtype=complex)
phase = np.exp(1j * np.pi / 4)   # global phase e^(i*pi/4)

psi_phased = phase * psi
print("Original probabilities:", np.abs(psi)**2)
print("Phased   probabilities:", np.abs(psi_phased)**2)
print("Probabilities unchanged:", np.allclose(np.abs(psi)**2, np.abs(psi_phased)**2))

In [ ]:
# Aggregate functions
probs = np.array([0.10, 0.45, 0.35, 0.10])
labels = np.array(["|00⟩", "|01⟩", "|10⟩", "|11⟩"])

print(f"Sum:     {probs.sum():.2f}")
print(f"Max:     {probs.max():.2f}  at {labels[probs.argmax()]}")
print(f"Min:     {probs.min():.2f}  at {labels[probs.argmin()]}")
print(f"Mean:    {probs.mean():.2f}")
print(f"Std dev: {probs.std():.4f}")

### Exercise 3.1

The quantum Fourier transform (QFT) amplitudes for an $n$-qubit system have the form:

$$\alpha_k = \frac{1}{\sqrt{N}} e^{2\pi i k j / N}, \quad N = 2^n$$

for a fixed input state $|j\rangle$.

For $n = 3$ and $j = 3$ (input state $|011\rangle$):
1. Compute all 8 QFT amplitudes as a NumPy array.
2. Compute and print the probabilities. What do you notice?
3. Verify normalization.
4. Compute the sum of all amplitudes. What is it for $j = 0$?

In [ ]:
import numpy as np

n = 3
N = 2**n
j = 3

# Your code here


---
## 4. Matrices and Matrix Multiplication

Quantum gates are unitary matrices. A 2D NumPy array is a matrix; `@` performs matrix multiplication.

In [ ]:
s = 1 / np.sqrt(2)

I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1,  0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[s, s], [s, -s]], dtype=complex)
S = np.array([[1, 0], [0, 1j]], dtype=complex)   # phase gate: sqrt(Z)

print("Hadamard gate:")
print(H.round(4))

In [ ]:
# Apply gates to |0⟩ and |1⟩
ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

print(f"H|0⟩ = {(H @ ket_0).round(4)}  (should be |+⟩)")
print(f"H|1⟩ = {(H @ ket_1).round(4)}  (should be |-⟩)")
print(f"X|0⟩ = {X @ ket_0}  (should be |1⟩)")
print(f"Z|1⟩ = {Z @ ket_1}  (should be -|1⟩)")

In [ ]:
# Compose gates: apply H then Z
ZH = Z @ H   # combined gate (applied right-to-left: H first, then Z)

# Both should give the same result
result_combined    = ZH @ ket_0
result_sequential  = Z @ (H @ ket_0)

print("Combined gate ZH|0⟩:", result_combined.round(4))
print("Sequential Z(H|0⟩): ", result_sequential.round(4))
print("Equal:", np.allclose(result_combined, result_sequential))

In [ ]:
# Verify unitarity: U† U = I
def is_unitary(U):
    """Return True if U is unitary within floating-point tolerance."""
    n = U.shape[0]
    return np.allclose(U.conj().T @ U, np.eye(n))

for name, gate in [("I", I), ("X", X), ("Y", Y), ("Z", Z), ("H", H), ("S", S)]:
    print(f"{name} is unitary: {is_unitary(gate)}")

In [ ]:
# Eigenvalues of quantum observables
# For Hermitian matrices (observables), use eigh (guarantees real eigenvalues)
# The Hamiltonian H = -Z has eigenvalues -1 (ground state) and +1 (excited state)

Ham = -Z   # simple Hamiltonian
eigenvalues, eigenvectors = np.linalg.eigh(Ham)

print("Energy eigenvalues:", eigenvalues)
print(f"Ground state (E={eigenvalues[0]:.0f}): {eigenvectors[:, 0]}")
print(f"Excited state (E={eigenvalues[1]:.0f}): {eigenvectors[:, 1]}")

In [ ]:
# Density matrix: |psi><psi|
psi = np.array([0.6, 0.8], dtype=complex)
rho = np.outer(psi, psi.conj())   # outer product

print("Density matrix rho = |psi><psi|:")
print(rho.round(4))
print(f"Trace(rho) = {np.trace(rho).real:.4f}  (must equal 1)")
print(f"Trace(rho^2) = {np.trace(rho @ rho).real:.4f}  (equals 1 for pure states)")

### Exercise 4.1

The Pauli matrices satisfy the relation $XYZ = iI$. Verify this numerically:

1. Compute $X @ Y @ Z$.
2. Use `np.allclose` to check whether the result equals $i \cdot I$.

Also verify the following commutation relations:
- $XY - YX = 2iZ$
- $YZ - ZY = 2iX$
- $ZX - XZ = 2iY$

In [ ]:
# Your code here


---
## 5. Reshaping and Slicing

In [ ]:
# 2D slicing
M = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

print("First row:   ", M[0])          # [1 2 3]
print("Second col:  ", M[:, 1])       # [2 5 8]
print("Element [1,2]:", M[1, 2])      # 6
print("Top-left 2x2:\n", M[:2, :2])   # [[1 2] [4 5]]

In [ ]:
# Boolean indexing: select elements satisfying a condition
probs  = np.array([0.05, 0.45, 0.35, 0.15])
labels = np.array(["|00⟩", "|01⟩", "|10⟩", "|11⟩"])

mask = probs > 0.1
print("Significant states:", labels[mask])
print("Their probabilities:", probs[mask])
print("Combined probability:", probs[mask].sum().round(4))

In [ ]:
# Reshape: change shape without copying data
n = 3
psi_flat = np.ones(2**n, dtype=complex) / np.sqrt(2**n)

print("Flat shape:", psi_flat.shape)         # (8,)

# Tensor form: one index per qubit
psi_tensor = psi_flat.reshape(2, 2, 2)
print("Tensor shape:", psi_tensor.shape)     # (2, 2, 2)

# Access amplitude for |q0=0, q1=1, q2=0>
print("Amplitude |010>:", psi_tensor[0, 1, 0].round(4))

# Flatten back
psi_back = psi_tensor.flatten()
print("Restored shape:", psi_back.shape)     # (8,)

In [ ]:
# Kronecker (tensor) product: combine systems
ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

# |01> = |0> tensor |1>
ket_01 = np.kron(ket_0, ket_1)
print("|01> =", ket_01)   # [0. 1. 0. 0.]

# |10> = |1> tensor |0>
ket_10 = np.kron(ket_1, ket_0)
print("|10> =", ket_10)   # [0. 0. 1. 0.]

# Bell state |Phi+> = (|00> + |11>) / sqrt(2)
phi_plus = (np.kron(ket_0, ket_0) + np.kron(ket_1, ket_1)) / np.sqrt(2)
print("|Phi+> =", phi_plus.round(4))

In [ ]:
# Kronecker product of gates: act on different qubits independently
# H on qubit 0, I on qubit 1: H tensor I
s = 1 / np.sqrt(2)
H = np.array([[s, s], [s, -s]], dtype=complex)
I = np.eye(2, dtype=complex)

HI = np.kron(H, I)   # 4x4 gate: H on qubit 0, identity on qubit 1
print("H tensor I (4x4):")
print(HI.round(4))

# Apply to |00>
ket_00 = np.kron(ket_0, ket_0)
result = HI @ ket_00
print("\n(H tensor I)|00> =", result.round(4))
print("Probabilities:", np.abs(result)**2.round(4))

### Exercise 5.1

Build the CNOT (controlled-NOT) gate as a 4x4 matrix:

$$\text{CNOT} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}$$

In the 2-qubit basis $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$, it flips the second qubit only when the first is $|1\rangle$.

1. Define CNOT as a NumPy array.
2. Verify it is unitary using `is_unitary` from earlier.
3. Apply CNOT to each of the four basis states and print the results.
4. Apply H to qubit 0 of $|00\rangle$ (use `np.kron(H, I)`), then apply CNOT. The result should be the Bell state $|\Phi^+\rangle$.

In [ ]:
import numpy as np

CNOT = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
], dtype=complex)

# Your code here


---
## Summary

| Operation | NumPy syntax | Notes |
|-----------|-------------|-------|
| Create array | `np.array([...], dtype=complex)` | Use `dtype=complex` for quantum states |
| Zeros / ones / identity | `np.zeros(n)`, `np.eye(n)` | Common starting points |
| Linspace / arange | `np.linspace(a, b, n)`, `np.arange(a, b)` | Evenly spaced grids |
| Shape, size, ndim | `.shape`, `.size`, `.ndim` | Array metadata |
| Normalization | `psi / np.linalg.norm(psi)` | Essential for quantum states |
| Probabilities | `np.abs(psi)**2` | Element-wise modulus squared |
| Inner product | `phi.conj() @ psi` | Conjugate bra, then dot |
| Gate application | `U @ psi` | Matrix-vector product |
| Gate composition | `U2 @ U1` | Applied right-to-left |
| Hermitian conjugate | `U.conj().T` | The dagger $U^\dagger$ |
| Unitarity check | `np.allclose(U.conj().T @ U, np.eye(n))` | Floating-point safe |
| Eigenvalues | `np.linalg.eigh(H)` | For Hermitian matrices |
| Density matrix | `np.outer(psi, psi.conj())` | Pure state $|\psi\rangle\langle\psi|$ |
| Reshape | `psi.reshape(2, 2, ..., 2)` | Tensor form: one index per qubit |
| Boolean index | `psi[probs > 0.1]` | Filter without loops |
| Kronecker product | `np.kron(A, B)` | Tensor product of states or gates |

**What comes next:** NumPy gives you the ability to compute with quantum states and gates. The next lesson introduces **Matplotlib**, which turns those numerical results into visualizations: probability histograms, bar charts of state amplitudes, and measurement outcome distributions.

---
## Challenge Problem

**Simulate the quantum Fourier transform on 3 qubits using matrix multiplication.**

The QFT matrix for $N = 2^n$ is:

$$\text{QFT}_{jk} = \frac{1}{\sqrt{N}} e^{2\pi i jk / N}, \quad j, k = 0, 1, \ldots, N-1$$

1. Build the 8x8 QFT matrix for $n = 3$ using a NumPy expression (no Python loops in the final version).
   *Hint:* Use `np.outer(np.arange(N), np.arange(N))` to build the matrix of exponent indices $jk$.
2. Verify it is unitary.
3. Apply it to the computational basis state $|3\rangle = |011\rangle$ (the array with a 1 at index 3 and 0 elsewhere).
4. Print the resulting amplitudes and probabilities.
5. Apply the QFT twice (QFT @ QFT) and check what you get. Can you explain the result?

In [ ]:
import numpy as np

n = 3
N = 2**n

# Your code here
